In [1]:
import pandas as pd
from pathlib import Path
h1 = pd.read_csv("data/raw/H1.csv")
h2 = pd.read_csv("data/raw/H2.csv")
print("Input rows:", len(h1) + len(h2))


Input rows: 119390


In [2]:
import duckdb

con = duckdb.connect("hotel_pipeline.duckdb")

con.execute("INSTALL ducklake;")
con.execute("LOAD ducklake;")

print("DuckDB and DuckLake are ready!")

DuckDB and DuckLake are ready!


## Task 3: Bronze Layer — Raw Data Ingestion

The Bronze layer preserves the original hotel booking records from H1.csv and H2.csv without applying cleaning transformations.

Each record will include ingestion metadata:
- source_name: Original CSV filename
- ingested_at: Timestamp when the record was loaded
- load_id: Identifier for the ingestion batch
- source_row_number: Original row position in the source file

The ingestion process will also record the number of rows loaded from each source file.

In [3]:
from pathlib import Path

# Create storage directory
Path("lakehouse_data").mkdir(exist_ok=True)

# Attach DuckLake catalog with local storage
con.execute("""
ATTACH 'ducklake:hotel_catalog.duckdb'
AS hotel_lake
(DATA_PATH 'lakehouse_data/');
""")

# Create Bronze, Silver, and Gold schemas
con.execute("CREATE SCHEMA IF NOT EXISTS hotel_lake.bronze;")
con.execute("CREATE SCHEMA IF NOT EXISTS hotel_lake.silver;")
con.execute("CREATE SCHEMA IF NOT EXISTS hotel_lake.gold;")

print("DuckLake attached successfully!")
print("Bronze, Silver, and Gold schemas created!")

DuckLake attached successfully!
Bronze, Silver, and Gold schemas created!


In [4]:
import hashlib
from datetime import datetime, timezone

# Prepare the original source files
source_files = [
    ("H1.csv", h1),
    ("H2.csv", h2)
]

bronze_batches = []

for filename, source_df in source_files:
    file_path = Path("data/raw") / filename

    # Generate a checksum for each source file
    checksum = hashlib.sha256(
        file_path.read_bytes()
    ).hexdigest()

    batch = source_df.drop(columns=["HotelType"], errors="ignore").copy()

    # Add ingestion metadata
    batch["source_name"] = filename
    batch["source_row_number"] = range(1, len(batch) + 1)
    batch["ingested_at"] = datetime.now(timezone.utc)
    batch["load_id"] = checksum[:16]

    bronze_batches.append(batch)

# Combine both batches
bronze_df = pd.concat(bronze_batches, ignore_index=True)

print("Prepared Bronze rows:", len(bronze_df))

# Register DataFrame with DuckDB
con.register("bronze_input", bronze_df)

# Create the Bronze table in DuckLake
con.execute("""
CREATE TABLE IF NOT EXISTS hotel_lake.bronze.hotel_bookings
AS SELECT * FROM bronze_input
WHERE 1 = 0
""")

# Avoid loading an unchanged file twice
for filename, _ in source_files:
    con.execute("""
        INSERT INTO hotel_lake.bronze.hotel_bookings
        SELECT *
        FROM bronze_input
        WHERE source_name = ?
          AND NOT EXISTS (
              SELECT 1
              FROM hotel_lake.bronze.hotel_bookings
              WHERE source_name = ?
                AND load_id = (
                    SELECT MIN(load_id)
                    FROM bronze_input
                    WHERE source_name = ?
                )
          )
    """, [filename, filename, filename])

print("Bronze ingestion completed!")

Prepared Bronze rows: 119390
Bronze ingestion completed!


In [5]:
# Check total records
total = con.execute("""
SELECT COUNT(*)
FROM hotel_lake.bronze.hotel_bookings
""").fetchone()[0]

print("Total Bronze records:", total)

# Check records by source file
report = con.execute("""
SELECT
    source_name,
    load_id,
    COUNT(*) AS input_rows,
    MIN(ingested_at) AS ingested_at
FROM hotel_lake.bronze.hotel_bookings
GROUP BY source_name, load_id
ORDER BY source_name
""").df()

display(report)

# Check uniqueness of source row identifiers
unique_count = con.execute("""
SELECT COUNT(*)
FROM (
    SELECT source_name, load_id, source_row_number
    FROM hotel_lake.bronze.hotel_bookings
    GROUP BY source_name, load_id, source_row_number
)
""").fetchone()[0]

print("Unique source identifiers:", unique_count)

assert total == 119390, "Unexpected Bronze row count"
assert unique_count == total, "Duplicate source identifiers detected"

print("Bronze validation passed!")

Total Bronze records: 119390


,source_name,load_id,input_rows,ingested_at
0,H1.csv,6f38fda13794fd6c,40060,2026-10-12 14:19:59.563900+03:00
1,H2.csv,5107090ffdc5585a,79330,2026-10-12 14:19:59.700552+03:00


Unique source identifiers: 119390
Bronze validation passed!
